# Learning Log — Market Anomaly Detection (Weeks 1–2)

A record of where I got stuck, what I learned, and the habits I practiced —
kept so the lessons outlast the project.

---

## PART 1 — Where I Got Stuck (and how it resolved)

**Dual virtual environment (venv vs .venv)**
PyTorch installed into one environment, but the notebook ran in another, so
`import torch` failed. Fix: point VS Code's interpreter at the right venv; keep
ONE venv named `.venv`. Lesson: when a "already installed" package isn't found,
run `where.exe python` and `pip show <pkg>` — the problem is almost always which
environment is active, not the package.

**PowerShell blocked the venv activation script (execution policy)**
`Set-ExecutionPolicy RemoteSigned -Scope CurrentUser` — one-time fix. Lesson:
Windows blocks scripts by default; this is expected, not a broken setup.

**pd.read_html failed twice: missing lxml, then 403 Forbidden**
lxml is a sub-dependency pandas needs to parse HTML tables. Then Wikipedia
blocked the automated request. Fix: fetch with a browser-like User-Agent header,
then parse. Lesson: read the BOTTOM line of a traceback first — it names the real
error. A *changed* error means the last fix worked; you advanced.

**Ticker format bug (silent, no crash)**
Wikipedia writes BRK.B; Yahoo wants BRK-B. Feeding dotted tickers returns empty
data with NO error. Fix: `.str.replace(".", "-", regex=False)` — and `regex=False`
matters, or the dot is treated as a wildcard and mangles everything. Lesson: the
most dangerous bugs don't crash — they silently return wrong data. Verify with a
check, don't trust that "it ran."

**The 2020 flagging mystery (a "bug" that was actually a finding)**
Expected the COVID crash to produce the most flags; it produced almost none at
its peak. Not a bug — the rolling z-score measures surprise RELATIVE to recent
volatility, and by April 2020 big moves had become the recent norm. Lesson: when
a result defies expectation, don't assume error — investigate. Sometimes the
"bug" is the discovery.

---

## PART 2 — Concepts I Learned (the durable knowledge)

**Virtual environments** — isolate each project's packages so version conflicts
between projects can't happen, and so `requirements.txt` reproducibly rebuilds
the exact environment. Install → re-freeze → commit.

**Log returns vs simple returns** — logs are time-additive (multi-day return =
sum of daily log returns) and symmetric (a doubling and a halving are equal and
opposite). Quant standard.

**Stationarity** — raw prices drift (non-stationary), so there's no stable
"normal" to model. Returns are ~stationary (stable mean near zero), which is what
makes anomaly detection possible. Model the *change*, not the *level*.

**Rolling / windowed statistics** — compute mean and std over a trailing window
so "normal" adapts to changing conditions AND never uses future data
(avoids look-ahead bias).

**Z-score as an anomaly signal** — (value − rolling mean) / rolling std =
"how many standard deviations from recent normal." The ±3σ rule is just a
threshold on this.

**Fat tails (leptokurtosis)** — real returns have far more extreme days than a
normal distribution predicts. Confirmed empirically: 0.70% flagged vs 0.3%
predicted. This is WHY the simple normality-based rule is limited.

**Right-skew and the log fix** — volume is right-skewed (floored at zero,
unbounded above), so z-scoring it raw gives lopsided results. Log-transform
FIRST to make it symmetric, then z-score. Same trick as log returns.

**MultiIndex / wide-vs-long data** — wide (days × tickers) is for computing;
long (one row per stock-day) is what models consume. `.stack()` + `pd.concat`
converts wide → long.

**Handling missing data — three actions, one principle:**
keep partial-history stocks (real signal), never fill gaps (fabrication),
drop incomplete feature rows (unusable). Principle: never fabricate; let the
data's real structure decide what's usable.

---

## PART 3 — Research Instincts I Practiced (the transferable skills)

- **Verify, don't trust "it ran."** Every fix confirmed with a check
  (list comprehension for the ticker fix, `.isna().sum()` for missing data,
  first-valid-date for the XYZ mystery). Output looking fine ≠ output being right.

- **Read data as a story, not a table.** `.describe()` and `.shape` each told me
  something — near-zero return mean = stationarity; all-positive volatility =
  magnitude has no direction; 223 missing days ≠ a 2025 listing.

- **Predict before running.** Guessing the result first (flag rate > 0.3%,
  volatility all-positive, z-score mean ~0) turns every run into a test of
  understanding, not just execution.

- **A surprising result is a lead, not a failure.** The April 2020 anomaly
  became the project's first real finding by investigating instead of dismissing.

- **Question authority, including AI.** Caught a fabricated citation, stale
  "use Python 3.11" advice, a nightly-vs-stable trap, and a Khan Academy resource
  that didn't exist. Stale/wrong claims from any source; verify.

- **Build simplest-thing-first.** Ran 10 tickers before 503; built the dumb ±3σ
  detector before the sophisticated ones. Small working version first, then scale.

---

## Key Finding So Far (worth remembering)
The ±3σ baseline detects *novelty relative to recent behavior*, not raw
magnitude. It flags regime *transitions* (crash onset) and misses sustained
crises (April 2020: 1 flag), because rolling normalization absorbs sustained
extremes into the baseline. This limitation is the reason the joint-feature
(Isolation Forest) and temporal (LSTM) detectors exist.